# 00 Env Sanity And Pinning

Local-only environment validation for Lumis-1.

Hard checks:
- `transformers` major version must be `5`
- `unsloth`, `unsloth_zoo`, and `datasets` must import
- GPU details must be captured

Required outputs:
- `workspace/reports/env_sanity.json`
- `workspace/reports/env_freeze.txt`

RunPod/Jupyter install reference:

```bash
python -m pip install -U unsloth unsloth_zoo datasets "transformers>=5,<6"
```


In [ ]:

from __future__ import annotations

import importlib
import importlib.metadata
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
REPORTS = REPO_ROOT / "workspace" / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

AUTO_INSTALL = False
INSTALL_PACKAGES = [
    "unsloth",
    "unsloth_zoo",
    "datasets",
    "transformers>=5,<6",
]

if AUTO_INSTALL:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-U", *INSTALL_PACKAGES],
        check=True,
        text=True,
    )

env = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "cwd": str(REPO_ROOT),
    "auto_install": AUTO_INSTALL,
    "install_packages": INSTALL_PACKAGES,
}

transformers_version = importlib.metadata.version("transformers")
transformers_major = int(transformers_version.split(".")[0])
env["transformers_version"] = transformers_version
env["transformers_major"] = transformers_major
if transformers_major != 5:
    raise RuntimeError(
        f"transformers major version must be 5 for Qwen3.5 + Unsloth. Found: {transformers_version}"
    )

for pkg in ("unsloth", "unsloth_zoo", "datasets"):
    importlib.import_module(pkg)
    env[f"{pkg}_import"] = True

gpu_info = {
    "cuda_available": False,
    "device_count": 0,
    "devices": [],
}
try:
    import torch

    gpu_info["cuda_available"] = bool(torch.cuda.is_available())
    gpu_info["device_count"] = int(torch.cuda.device_count())
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            gpu_info["devices"].append(
                {
                    "index": i,
                    "name": torch.cuda.get_device_name(i),
                    "total_memory_gb": round(props.total_memory / (1024**3), 2),
                    "capability": f"{props.major}.{props.minor}",
                }
            )
except Exception as exc:  # noqa: BLE001
    gpu_info["torch_probe_error"] = str(exc)

env["gpu"] = gpu_info

env_sanity_path = REPORTS / "env_sanity.json"
env_sanity_path.write_text(json.dumps(env, indent=2), encoding="utf-8")

freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True,
    text=True,
    check=True,
)
(REPORTS / "env_freeze.txt").write_text(freeze.stdout, encoding="utf-8")

print(json.dumps(env, indent=2))
print(f"saved: {env_sanity_path}")
print(f"saved: {REPORTS / 'env_freeze.txt'}")
